# Neural network #
1D convolutional network (PyTorch) on sliding windows: the consumption of the 168 hours ending 145 hours before the target hour and the temperature forecast for the 168 hours up to it, plus location, hour and weekday. Same test period as `xgboost.ipynb`, tuned with Optuna on three rolling validation windows at the end of the training data, so the test set is only used once at the end. The final forecast is the mean of networks trained with different seeds.

Requires `pip install torch optuna`.

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
import optuna
from forecast_checks import LeakageChecker, ForecastValidator, MakeTrainingData
from neural_network_helper import SEED, MAX_EPOCHS, WindowScaler, train_model, ensemble_forecast

torch.manual_seed(SEED)
np.random.seed(SEED)
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
SCOORING_METRIC="root_mean_square_error"

In [ ]:
#One week of hours covers both the daily and the weekly cycle
WINDOW_HOURS = 168
trainer = MakeTrainingData(window_hours=WINDOW_HOURS)
df = trainer.df

In [ ]:
#Leakage check of the windows: scramble everything unknown when a day is forecast and check that its windows do not change.
#Random days, plus the week after the October DST change, where a consumption window starting 144 h back would leak
checker = LeakageChecker(df[["location", "time", "consumption", "temperature"]])
build_windows = lambda raw: MakeTrainingData.add_sliding_window(raw, WINDOW_HOURS)
checker.check_features(build_windows)
checker.check_features(build_windows, target_days=pd.date_range("2022-10-30", "2022-11-05"))

In [ ]:
#Same temporal split as xgboost.ipynb; the row labels of the split index trainer.df
data = trainer.get_train_test()
train, test = (df.loc[data[part]["feature"].index] for part in ("train", "test"))
LeakageChecker.check_split(train["date"], test["date"])

#Optuna tunes on three consecutive 2-week windows at the end of the training period (late autumn and winter, like the test period).
#Each window is trained on everything up to 6 days before it; several windows make the choice of model less dependent on one period
VAL_DAYS, N_FOLDS = 14, 3
folds = []
for k in reversed(range(N_FOLDS)):
    val_end = train["date"].max() - pd.Timedelta(days=VAL_DAYS * k)
    val_start = val_end - pd.Timedelta(days=VAL_DAYS - 1)
    fit = train[train["date"] < val_start - pd.Timedelta(days=6)]
    val = train[(train["date"] >= val_start) & (train["date"] <= val_end)]
    LeakageChecker.check_split(fit["date"], val["date"])
    folds.append((fit, val))
    print(f"fit: {fit['date'].min().date()} - {fit['date'].max().date()}, val: {val['date'].min().date()} - {val['date'].max().date()}")

In [ ]:
feats = ["location", "hour", "weekday", *trainer.consumption_window, *trainer.temperature_window]
LeakageChecker.check_feature_names(feats)
LOCATIONS = sorted(df["location"].unique())

#The network cannot take missing values. Also require the point lags, so the test hours are the same as in xgboost.ipynb
scored = feats + ["consumption_6days_ago", "consumption_7days_ago", "consumption_14days_ago"]
train, test = (d.dropna(subset=scored) for d in (train, test))
folds = [(fit.dropna(subset=scored), val.dropna(subset=scored)) for fit, val in folds]


def make_scaler(data):
    return WindowScaler(data, trainer.consumption_window, trainer.temperature_window, LOCATIONS)

In [ ]:
#Tune on the validation windows; each window's scaler only sees its own fit period
fold_sets = []
for fit, val in folds:
    fit_scaler = make_scaler(fit)
    fold_sets.append((fit_scaler.X(fit), fit_scaler.y(fit), (fit_scaler.X(val), val["consumption"], fit_scaler, val)))
N_TRIALS = 40


def objective(trial):
    params = {"n_conv": trial.suggest_int("n_conv", 1, 3),
              "channels": trial.suggest_categorical("channels", [16, 32, 64]),
              "kernel": trial.suggest_categorical("kernel", [3, 5, 7]),
              "hidden": trial.suggest_categorical("hidden", [32, 64, 128]),
              "dropout": trial.suggest_float("dropout", 0.0, 0.3),
              "lr": trial.suggest_float("lr", 1e-4, 1e-2, log=True),
              "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
              "batch_size": trial.suggest_categorical("batch_size", [256, 512, 1024])}
    maes, epochs = [], []
    for step, (X_fit, y_fit, val_set) in enumerate(fold_sets):
        _, mae, best_epoch = train_model(params, X_fit, y_fit, MAX_EPOCHS, val=val_set)
        maes.append(mae)
        epochs.append(best_epoch)
        #Every trial sees the windows in the same order, so the running mean can be compared between trials
        trial.report(np.mean(maes), step)
        if trial.should_prune():
            raise optuna.TrialPruned()
    trial.set_user_attr("best_epoch", round(np.mean(epochs)))
    trial.set_user_attr("fold_maes", maes)
    return np.mean(maes)


start = time.time()
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED),
                            pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=0))
#Fixed number of trials, not a time budget: with a timeout the number of trials (and so the chosen model) depends on machine speed
study.optimize(objective, n_trials=N_TRIALS)
pruned = sum(t.state == optuna.trial.TrialState.PRUNED for t in study.trials)
print(f"{len(study.trials)} trials ({pruned} pruned) in {time.time() - start:.0f} s")
print("Best mean validation MAE:", study.best_value, "per window:", study.best_trial.user_attrs["fold_maes"],
      "at epoch", study.best_trial.user_attrs["best_epoch"])
study.best_params

In [ ]:
#Check the chosen model per location on the last validation window before touching the test set, so a location where it loses to the baseline shows up here
fit, val = folds[-1]
val_validator = (ForecastValidator(val["consumption"], groups=val["location"])
                 .add("Same hour last week", val["consumption_7days_ago"])
                 .add("Neural network", ensemble_forecast(study.best_params, study.best_trial.user_attrs["best_epoch"],
                                                          fit, val, make_scaler(fit))))
val_validator.per_group()

In [ ]:
#Refit on the whole training period with the best parameters and number of epochs, then predict the test period once
pred = ensemble_forecast(study.best_params, study.best_trial.user_attrs["best_epoch"], train, test, make_scaler(train))
print(f"Total training time: {time.time() - start:.0f} s")
test = test.assign(pred=pred)

In [ ]:
#Same hours as the comparison in xgboost.ipynb (test rows where every lag exists), so the tables can be compared directly
validator = (ForecastValidator(test["consumption"], groups=test["location"])
             .add("Same hour last week", test["consumption_7days_ago"], baseline=True)
             .add("Neural network", test["pred"]))
validator.summary()

In [ ]:
validator.save_to_json("results/neural_network.json")

In [ ]:
validator.per_group()

In [ ]:
validator.plot_predicted_vs_actual_consumption()